In [1]:
'''
Activate venv
    source /home/acrfa/ml-class/.venv/bin/activate
'''

'\nActivate venv\n    source /home/acrfa/ml-class/.venv/bin/activate\n'

In [2]:
'''
1. Obtain a Keras or Python implementation of an SSD or YOLO model and study how it is used for
object detection.
2. Develop a system that detects and tracks three kinds of target objects, such as cups, boxes, oranges, or
balls, in a live camera stream.
3. Capture or collect training images as needed and retrain or fine-tune the selected model when appro-
priate.
4. Continuously acquire live images, detect all target objects, and display the results with bounding boxes
overlaid on the frames.
Expected Outputs
1
Machine Learning for Robot Perception Project Brief
• A live on-screen detection and tracking view.
• Source code and any required model or configuration files needed to run the system.
Evaluation
Evaluation criterion Weight
Code and related files are submitted correctly 20 points
Code executes without syntax errors 30 points
A live window continuously shows detection and tracking results 30 points
Detection results are reasonably successful, above 80% accuracy 20 points
'''

'\n1. Obtain a Keras or Python implementation of an SSD or YOLO model and study how it is used for\nobject detection.\n2. Develop a system that detects and tracks three kinds of target objects, such as cups, boxes, oranges, or\nballs, in a live camera stream.\n3. Capture or collect training images as needed and retrain or fine-tune the selected model when appro-\npriate.\n4. Continuously acquire live images, detect all target objects, and display the results with bounding boxes\noverlaid on the frames.\nExpected Outputs\n1\nMachine Learning for Robot Perception Project Brief\n• A live on-screen detection and tracking view.\n• Source code and any required model or configuration files needed to run the system.\nEvaluation\nEvaluation criterion Weight\nCode and related files are submitted correctly 20 points\nCode executes without syntax errors 30 points\nA live window continuously shows detection and tracking results 30 points\nDetection results are reasonably successful, above 80% accur

In [3]:
'''
1. Collect images for your 3 classes (with variations and noise).
2. Label images with bounding boxes (YOLO format: class, x_center, y_center, width, height).
3. Train a YOLO model (v5/v8) using your dataset.
4. Export the trained YOLO weights.
5. In your live video loop:
     a. Read frame
     b. Feed frame into YOLO model
     c. Get bounding boxes and class predictions
     d. Draw boxes on frame
     e. Display frame

'''

'\n1. Collect images for your 3 classes (with variations and noise).\n2. Label images with bounding boxes (YOLO format: class, x_center, y_center, width, height).\n3. Train a YOLO model (v5/v8) using your dataset.\n4. Export the trained YOLO weights.\n5. In your live video loop:\n     a. Read frame\n     b. Feed frame into YOLO model\n     c. Get bounding boxes and class predictions\n     d. Draw boxes on frame\n     e. Display frame\n\n'

In [31]:
#Project by Lauren Nunez, Saam Grami, Regina Martinez
 
import cv2
import torch
from ultralytics import YOLO
import tensorflow as tf
import roboflow
from pathlib import Path

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
#This is camera code 
import numpy as np
import cv2 
import matplotlib.pyplot as plt

np.set_printoptions(threshold=np.inf)
cap = cv2.VideoCapture(0)

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('live_camera.avi', fourcc, 60.0, (640,  480))

while True:
    ret, frame = cap.read()
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break
    # frame = cv2.flip(frame, 0)
    
    # write the flipped frame
    out.write(frame)
    counter= counter +1
    #cv2.imshow('frame', frame)
    if cv2.waitKey(1) == ord('q'):
        break
    print(counter)

   
   
        
        



# Release everything if job is finished
cap.release()
out.release()
cv2.destroyAllWindows()


1
2
3
4


In [ ]:
#YOLO data prep
'''
YOLO requires for each image that you have the image and a text file 
    that goes with it that has the bounding boxes

YOLO versions take the same .txt and image files for their input
'''

In [ ]:
#This code takes edge impulse exported data, which is a json line
#Then it takes the json line and creates .txt file that can accompany the photos


import json
import os
from PIL import Image

def convert(folder):
    labels_file = os.path.join(folder, "bounding_boxes.labels")
    images_out = os.path.join(folder, "images")
    labels_out = os.path.join(folder, "labels")
    os.makedirs(images_out, exist_ok=True)
    os.makedirs(labels_out, exist_ok=True)

    with open(labels_file) as f:
        data = json.load(f)

    labels_dict = data["boundingBoxes"]

    # Build class list
    all_labels = sorted(set(
        bb["label"]
        for img_name, boxes in labels_dict.items()
        for bb in boxes
    ))
    class_map = {label: i for i, label in enumerate(all_labels)}

    for img_name, boxes in labels_dict.items():
        img_path = os.path.join(folder, img_name)

        if not os.path.exists(img_path):
            print(f"Missing image: {img_path}")
            continue

        with Image.open(img_path) as img:
            w, h = img.size

        os.rename(img_path, os.path.join(images_out, img_name))

        stem = os.path.splitext(img_name)[0]
        txt_path = os.path.join(labels_out, stem + ".txt")
        with open(txt_path, "w") as out:
            for bb in boxes:
                cls = class_map[bb["label"]]
                cx = (bb["x"] + bb["width"] / 2) / w
                cy = (bb["y"] + bb["height"] / 2) / h
                bw = bb["width"] / w
                bh = bb["height"] / h
                out.write(f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

    return all_labels

train_classes = convert("bible-recognizer-export/training")
test_classes  = convert("bible-recognizer-export/testing")

all_classes = sorted(set(train_classes + test_classes))

with open("bible-recognizer-export/classes.txt", "w") as f:
    f.write("\n".join(all_classes))

with open("bible-recognizer-export/data.yaml", "w") as f:
    f.write(f"train: training/images\n")
    f.write(f"val: testing/images\n")
    f.write(f"nc: {len(all_classes)}\n")
    f.write(f"names: {all_classes}\n")

print("Done! Classes:", all_classes)

Done! Classes: ['Bible']


In [13]:
# 1. Import the library
from inference_sdk import InferenceHTTPClient

# 2. Connect to your workspace
client = InferenceHTTPClient(
  api_url="https://serverless.roboflow.com",
  api_key="vi1TgyNxg7symKVltpBB"
)

# 3. Run your workflow on an image
result = client.run_workflow(
  workspace_name="saam-haghighat-grami",
  workflow_id="general-segmentation-api",
  images={
    "image": "Screenshot 2026-04-02 174314.png"  # Path to your image file
  },
  parameters={
    "classes": "glasses, Glasses"
  },
  use_cache=True  # cache workflow definition for 15 minutes
)

# 4. Get your results
print(result)

import base64

# Get the string
img_base64 = result[0]["annotated_image"]

# Decode
img_bytes = base64.b64decode(img_base64)

# Save to file
with open("output.jpg", "wb") as f:
    f.write(img_bytes)

print("Saved as output.jpg")

[{'annotated_image': '...', 'predictions': {'image': {'width': 704, 'height': 364}, 'predictions': [{'width': 608.0, 'height': 206.0, 'x': 356.0, 'y': 207.0, 'confidence': 0.9765625, 'class_id': 0, 'points': [{'x': 516.0, 'y': 104.0}, {'x': 515.0, 'y': 105.0}, {'x': 501.0, 'y': 105.0}, {'x': 500.0, 'y': 106.0}, {'x': 490.0, 'y': 106.0}, {'x': 489.0, 'y': 107.0}, {'x': 480.0, 'y': 107.0}, {'x': 479.0, 'y': 108.0}, {'x': 472.0, 'y': 108.0}, {'x': 471.0, 'y': 109.0}, {'x': 464.0, 'y': 109.0}, {'x': 463.0, 'y': 110.0}, {'x': 459.0, 'y': 110.0}, {'x': 458.0, 'y': 111.0}, {'x': 454.0, 'y': 111.0}, {'x': 453.0, 'y': 112.0}, {'x': 449.0, 'y': 112.0}, {'x': 448.0, 'y': 113.0}, {'x': 445.0, 'y': 113.0}, {'x': 444.0, 'y': 114.0}, {'x': 441.0, 'y': 114.0}, {'x': 440.0, 'y': 115.0}, {'x': 438.0, 'y': 115.0}, {'x': 437.0, 'y': 116.0}, {'x': 435.0, 'y': 116.0}, {'x': 434.0, 'y': 117.0}, {'x': 432.0, 'y': 117.0}, {'x': 431.0, 'y': 118.0}, {'x': 429.0, 'y': 118.0}, {'x': 428.0, 'y': 119.0}, {'x': 426.0

In [38]:

# Glasses model

from ultralytics import YOLO
model = YOLO("yolo26s.pt")
#model = YOLO("yolo26x-seg.yaml")
# Train the model
results = model.train(data=r"C:\Users\acrfa\Downloads\Glasses.v1i.yolo26\data.yaml", 
                       epochs=1,
                       fraction=0.1,
    imgsz=320,   # Half the size = ~4x faster
    batch=4,
    workers=2,
    device="cpu"
)


Ultralytics 8.4.33  Python-3.12.10 torch-2.11.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\acrfa\Downloads\Glasses.v1i.yolo26\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.1, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train10, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overla

In [35]:
# Glasses model

from ultralytics import YOLO
model = YOLO("yolo26s.pt")
#model = YOLO("yolo26x-seg.yaml")
# Train the model
results = model.train(data=r"C:\Users\acrfa\Downloads\Glasses.v1i.yolo26\data.yaml", 
                       epochs=5,
    imgsz=320,   # Half the size = ~4x faster
    batch=4,
    workers=2,
    device="cpu"
)

Ultralytics 8.4.33  Python-3.12.10 torch-2.11.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\acrfa\Downloads\Glasses.v1i.yolo26\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

KeyboardInterrupt: 

In [ ]:
#tells file if files are open
import os

file_path = r"C:\Users\acrfa\OneDrive\Classes\#2026 Spring KSU\MTRE 4820 Machine Learning\Glasses.v1i.yolo26"
print(os.path.exists(file_path))
print(os.access(file_path, os.R_OK))  # True means readable

False
False


In [20]:
from ultralytics import YOLO
import os
dataset_yaml = r"C:\YOLO\Glasses.v1i.yolo26"

# Check that the file exists and is readable
if not os.path.exists(dataset_yaml):
    raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")
if not os.access(dataset_yaml, os.R_OK):
    raise PermissionError(f"Cannot read dataset YAML: {dataset_yaml}")

FileNotFoundError: Dataset YAML not found: C:\YOLO\Glasses.v1i.yolo26

In [34]:
import torch
print(torch.cuda.is_available())       # Should be True
print(torch.cuda.get_device_name(0))   # Should show your GPU name


False


AssertionError: Torch not compiled with CUDA enabled

In [33]:


# Using pre-trained YOLO Model with already defined weights and layers and all that business
# Reference: https://docs.ultralytics.com/models/yolo26/#usage-examples
 
phone_yaml = Path.home() / "Downloads/mobilephone.yolo26/data"
headphones_yaml = Path.home() / "Downloads/EarphoneDetection.v1i.yolo26/data"
glasses_yaml = Path.home() / "Downloads/Glasses.v1i.yolo26/data"
 
# Load a COCO-pretrained YOLO26n model
model_phone = YOLO("yolo26n.pt")
model_headphones = YOLO("yolo26n.pt")
model_glasses = YOLO("yolo26n.pt")
 
# Reference: https://docs.ultralytics.com/modes/train/
# Train the model. Change to whaterver the name for the .yaml file is..
results_phone = model_phone.train(data=phone_yaml, epochs=100, imgsz=640)
results_headphones = model_glasses.train(data=headphones_yaml, epochs=100, imgsz=640)
results_glasses = model_headphones.train(data=glasses_yaml, epochs=100, imgsz=640)

NameError: name 'Path' is not defined

In [39]:

from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

RUN_DIR = Path(r"C:\Users\acrfa\OneDrive - Kennesaw State University\Classes\#2026 Spring KSU\MTRE 4820 Machine Learning\MTRE 4820 Virtual Environment\mtre4820_project5\runs\detect\train10")
WEIGHTS = RUN_DIR / "weights" / "best.pt"

model = YOLO(str(WEIGHTS))

# ─── Run inference ────────────────────────────────────────────────────────────
results = model.predict(
    source=r"C:\path\to\your\test\images",  # ← change this
    conf=0.25,
    imgsz=320,
    save=True,   # saves annotated images to disk
)

# ─── Display the saved images with boxes ─────────────────────────────────────
saved_dir = Path(results[0].save_dir)
image_files = list(saved_dir.glob("*.jpg")) + list(saved_dir.glob("*.png"))

print(f"Showing {len(image_files)} images from {saved_dir}")

for img_path in image_files:
    img = mpimg.imread(img_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

FileNotFoundError: C:\path\to\your\test\images does not exist